# CoalGameRec — Round 9: run ALL remaining experiments

Runs every still-missing experiment from the round-9 deep review. **Everything is resume-safe**:
attribution checkpoints every 25 users, per-seed skips, and skip-if-done guards in every script.
Interrupt and re-run any cell at any time — it continues where it left off.

**Recommended order (Part A = mandatory for Accept, ~8 h on M-series MPS):**

| Cell | Experiment | Review item | Est. time |
|---|---|---|---|
| A1 | v7 corrected protocol — Amazon | #1 temporal protocol | ~45 min |
| A2 | Matched λ sweep — Amazon | #4 within-model curves | ~50 min |
| A3 | Sequential baselines — Amazon (3 seeds) + ML-1M (2 seeds) | #10 stronger baselines | ~3 h |
| A4 | Controlled randomization — Amazon + ML-1M | #11 weights-only swaps | ~2.5 h |
| A5 | Nested leak-free λ tuning — Amazon (3) + ML-1M (2) | #3 replaces circular table | ~5.5 h |
| B1 | Selection×valuation factorial — Amazon (2) + ML-1M (1) | #2 | ~2.5 h |
| B2 | Convergence v2 — Amazon + ML-1M | #8 M=1024 reference | ~4 h |
| B3 | Negset multi-draw — Amazon ×2 (+ML-1M optional) | #12 | ~2 h (+5 h) |
| B4 | Masked-forward multi-seed — Amazon 43–46 + ML-1M 43–46 | faithfulness curves | ~5 h |
| C1 | Design ablations — Amazon seeds 43–44 | #7 multi-seed + intervention | ~8 h |
| C2 | Design ablations — ML-1M seeds 43–44 | #7 (overnight) | ~20–24 h |
| ALL | Run everything above sequentially | — | ~40 h |

Logs stream into `results/journal_runs/_notebook_logs/r9_*.log`. After each part (or at the end),
run the **PUSH** cell and paste the results back.


In [ ]:
# Cell 1 — Setup: paths, device detection, streaming helper
from pathlib import Path
import os, sys, subprocess, platform

CWD = Path.cwd().resolve()
CODE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
RESULTS = CODE_DIR / "results" / "journal_runs"
LOGDIR = RESULTS / "_notebook_logs"
LOGDIR.mkdir(parents=True, exist_ok=True)
V3M = RESULTS / "ml1m_lightgcn_v3_prospective"
V3A = RESULTS / "amazon_books_lightgcn_v3_prospective"

import torch
if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

def stream(cmd, log, env=None):
    """Run cmd, stream output live into the notebook AND tee to a persistent log."""
    e = dict(os.environ, COALGAME_DEVICE=DEVICE)
    if env: e.update(env)
    print("RUN:", " ".join(str(c) for c in cmd))
    lf = open(LOGDIR / log, "a")
    lf.write("\n$ " + " ".join(str(c) for c in cmd) + "\n"); lf.flush()
    with subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          cwd=str(CODE_DIR), env=e, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="")
            lf.write(line); lf.flush()
    lf.close()
    print("exit code:", p.returncode)
    return p.returncode

PY = sys.executable
print("CODE_DIR:", CODE_DIR)
print("device  :", DEVICE, "| torch", torch.__version__, "| python", platform.python_version())


In [ ]:
# Cell 2 — Environment check (scripts + source runs present?)
required_scripts = ["run_protocol_v7.py", "run_matched_lambda_sweep.py", "run_sequential_baselines.py",
                    "run_controlled_randomization.py", "run_nested_lambda_tuning.py",
                    "run_selection_factorial.py", "run_convergence_v2.py", "run_negset_sensitivity.py",
                    "run_design_ablations.py", "run_masked_forward_faithfulness.py"]
required_runs = {"ML-1M v3 run": V3M / "splits" / "train.parquet",
                 "Amazon v3 run": V3A / "splits" / "train.parquet",
                 "ML-1M item-vector report": V3M / "item_vectors_report.json",
                 "Amazon item-vector report": V3A / "item_vectors_report.json"}
ok = True
for s in required_scripts:
    p = CODE_DIR / "scripts" / s
    print(f"{'OK ' if p.exists() else 'MISSING'} script: {s}")
    ok = ok and p.exists()
for name, path in required_runs.items():
    print(f"{'OK ' if path.exists() else 'MISSING'} {name}")
    ok = ok and path.exists()
print("\nReady." if ok else "\nFix missing pieces before running (git pull first).")


In [ ]:
# ===== PART A: MANDATORY FOR ACCEPT (~8 h) =====
# A1 — v7 corrected protocol, Amazon (review item #1).
# Calibration item excluded from candidates; z-scores over corrected set; 9 families, 5 seeds.
rc = stream([PY, "scripts/run_protocol_v7.py", "--dataset", "amazon",
             "--source-run", V3A, "--out", RESULTS / "amazon_books_lightgcn_v7_corrected_protocol"],
            "r9_v7_amazon.log", env={"C1_WITH_SHAPLEY": "1"})
assert rc == 0, "v7 amazon failed"
print("A1 DONE: amazon_books_lightgcn_v7_corrected_protocol")


In [ ]:
# A2 — Matched single-execution lambda sweep, Amazon (review item #4).
# All 8 families on identical fitted models, per-user metrics per lambda.
rc = stream([PY, "scripts/run_matched_lambda_sweep.py", "--dataset", "amazon",
             "--source-run", V3A, "--out", RESULTS / "amazon_books_lightgcn_v8_matched_lambda_sweep",
             "--seeds", "42", "43", "44", "45", "46"],
            "r9_matchedsweep_amazon.log", env={"C1_WITH_SHAPLEY": "1"})
assert rc == 0, "matched sweep amazon failed"
print("A2 DONE: amazon_books_lightgcn_v8_matched_lambda_sweep")


In [ ]:
# A3 — Sequential same-information baselines (review item #10):
# last-item kNN, updated profile, frozen-graph edge update, recency. Amazon 3 seeds, ML-1M 2 seeds.
rc = stream([PY, "scripts/run_sequential_baselines.py", "--dataset", "amazon",
             "--source-run", V3A, "--out", RESULTS / "amazon_books_lightgcn_v8_sequential_baselines",
             "--seeds", "42", "43", "44"], "r9_seqbaselines_amazon.log")
assert rc == 0, "sequential baselines amazon failed"
rc = stream([PY, "scripts/run_sequential_baselines.py", "--dataset", "ml1m",
             "--source-run", V3M, "--out", RESULTS / "ml1m_lightgcn_v8_sequential_baselines",
             "--seeds", "42", "43"], "r9_seqbaselines_ml1m.log")
assert rc == 0, "sequential baselines ml1m failed"
print("A3 DONE: sequential baselines (both datasets)")


In [ ]:
# A4 — Controlled randomization sanity (review item #11):
# trained scorer/embeddings fixed; only attribution weights swapped
# (untrained-model weights, shuffled, distribution-matched, selection-only).
rc = stream([PY, "scripts/run_controlled_randomization.py", "--dataset", "amazon",
             "--source-run", V3A, "--out", RESULTS / "amazon_books_lightgcn_v8_controlled_randomization"],
            "r9_ctrlrand_amazon.log")
assert rc == 0, "controlled randomization amazon failed"
rc = stream([PY, "scripts/run_controlled_randomization.py", "--dataset", "ml1m",
             "--source-run", V3M, "--out", RESULTS / "ml1m_lightgcn_v8_controlled_randomization"],
            "r9_ctrlrand_ml1m.log")
assert rc == 0, "controlled randomization ml1m failed"
print("A4 DONE: controlled randomization (both datasets)")


In [ ]:
# A5 — Nested leak-free lambda tuning (review item #3):
# signal from event t-2, lambda tuned predicting event t-1, final signal from t-1 predicting t.
# Replaces the circular exploratory table.
rc = stream([PY, "scripts/run_nested_lambda_tuning.py", "--dataset", "amazon",
             "--source-run", V3A, "--out", RESULTS / "amazon_books_lightgcn_v8_nested_tuning",
             "--seeds", "42", "43", "44"], "r9_nested_amazon.log")
assert rc == 0, "nested tuning amazon failed"
rc = stream([PY, "scripts/run_nested_lambda_tuning.py", "--dataset", "ml1m",
             "--source-run", V3M, "--out", RESULTS / "ml1m_lightgcn_v8_nested_tuning",
             "--seeds", "42", "43"], "r9_nested_ml1m.log")
assert rc == 0, "nested tuning ml1m failed"
print("A5 DONE: nested tuning (both datasets)")


In [ ]:
# ===== PART B: QUEUED REPLICATIONS (~13.5 h) =====
# B1 — 2x2 selection x valuation factorial (review item #2).
rc = stream([PY, "scripts/run_selection_factorial.py", "--dataset", "amazon",
             "--source-run", V3A, "--out", RESULTS / "amazon_books_lightgcn_v8_selection_factorial",
             "--seeds", "42", "43"], "r9_factorial_amazon.log")
assert rc == 0, "factorial amazon failed"
rc = stream([PY, "scripts/run_selection_factorial.py", "--dataset", "ml1m",
             "--source-run", V3M, "--out", RESULTS / "ml1m_lightgcn_v8_selection_factorial",
             "--seeds", "42"], "r9_factorial_ml1m.log")
assert rc == 0, "factorial ml1m failed"
print("B1 DONE: selection x valuation factorial")


In [ ]:
# B2 — Shapley convergence v2 (review item #8): independent M=1024 reference,
# exact Shapley for small games, attribution error / sign / top-12, downstream NDCG vs M.
rc = stream([PY, "scripts/run_convergence_v2.py", "--dataset", "amazon",
             "--source-run", V3A, "--out", RESULTS / "amazon_books_lightgcn_v8_convergence_v2",
             "--max-users", "1000"], "r9_convergence_amazon.log")
assert rc == 0, "convergence amazon failed"
rc = stream([PY, "scripts/run_convergence_v2.py", "--dataset", "ml1m",
             "--source-run", V3M, "--out", RESULTS / "ml1m_lightgcn_v8_convergence_v2",
             "--max-users", "1000"], "r9_convergence_ml1m.log")
assert rc == 0, "convergence ml1m failed"
print("B2 DONE: convergence v2 (both datasets)")


In [ ]:
# B3 — Validation-negative multi-draw sensitivity (review item #12):
# same trained model, two independent negative draws (offsets 1e6, 2e6) at sizes 50/100/500.
# Amazon mandatory; ML-1M optional (uncomment).
for OFF in ["1000000", "2000000"]:
    rc = stream([PY, "scripts/run_negset_sensitivity.py", "--dataset", "amazon",
                 "--source-run", V3A,
                 "--out", RESULTS / f"amazon_books_lightgcn_v6_negset_sensitivity_draw_{OFF}",
                 "--sizes", "50", "100", "500", "--max-users", "1500", "--neg-offset", OFF],
                f"r9_negset_draw_{OFF}_amazon.log")
    assert rc == 0, f"negset draw {OFF} amazon failed"
# for OFF in ["1000000", "2000000"]:   # optional ML-1M draws (~2.5 h each)
#     rc = stream([PY, "scripts/run_negset_sensitivity.py", "--dataset", "ml1m",
#                  "--source-run", V3M,
#                  "--out", RESULTS / f"ml1m_lightgcn_v6_negset_sensitivity_draw_{OFF}",
#                  "--sizes", "50", "100", "500", "--max-users", "1500", "--neg-offset", OFF],
#                 f"r9_negset_draw_{OFF}_ml1m.log")
#     assert rc == 0
print("B3 DONE: negset multi-draw (amazon)")


In [ ]:
# B4 — Masked-forward faithfulness, multi-seed (faithfulness curves with 3+ seeds):
# Amazon seeds 43-46 and ML-1M seeds 43-46 (seed 42 already released for both).
for ds, seeds in [("amazon", ["43", "44", "45", "46"]), ("ml1m", ["43", "44", "45", "46"])]:
    for s in seeds:
        rc = stream([PY, "scripts/run_masked_forward_faithfulness.py", "--dataset", ds,
                     "--seed", s, "--n-users", "1000"],
                    f"r9_maskedfwd_{ds}_{s}.log", env={"COALGAME_DEVICE": "cpu"})
        assert rc == 0, f"masked-forward {ds} seed {s} failed"
print("B4 DONE: masked-forward multi-seed")


In [ ]:
# ===== PART C: LONG TAIL (overnight+) =====
# C1 — Multi-seed design ablations, Amazon seeds 43-44 (review item #7;
# includes the native-vs-external-kernel intervention rows).
for s in ["43", "44"]:
    rc = stream([PY, "scripts/run_design_ablations.py", "--dataset", "amazon", "--seed", s],
                f"r9_designabl_amazon_{s}.log")
    assert rc == 0, f"design ablations amazon seed {s} failed"
print("C1 DONE: design ablations amazon seeds 43-44")


In [ ]:
# C2 — Multi-seed design ablations, ML-1M seeds 43-44 (~10-12 h EACH; run overnight).
for s in ["43", "44"]:
    rc = stream([PY, "scripts/run_design_ablations.py", "--dataset", "ml1m", "--seed", s],
                f"r9_designabl_ml1m_{s}.log")
    assert rc == 0, f"design ablations ml1m seed {s} failed"
print("C2 DONE: design ablations ml1m seeds 43-44")


In [ ]:
# ===== RUN ALL =====
# Everything above, sequentially (Parts A -> B -> C). Fully resume-safe:
# if interrupted, re-run this cell — finished seeds/stages skip automatically.
import subprocess, os, sys

STAGES = [
    # (name, log, env, cmd)
    ("A1 v7 amazon", "r9_v7_amazon.log", {"C1_WITH_SHAPLEY": "1"},
     [PY, "scripts/run_protocol_v7.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v7_corrected_protocol"]),
    ("A2 matched sweep amazon", "r9_matchedsweep_amazon.log", {"C1_WITH_SHAPLEY": "1"},
     [PY, "scripts/run_matched_lambda_sweep.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v8_matched_lambda_sweep",
      "--seeds", "42", "43", "44", "45", "46"]),
    ("A3 seq baselines amazon", "r9_seqbaselines_amazon.log", {},
     [PY, "scripts/run_sequential_baselines.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v8_sequential_baselines", "--seeds", "42", "43", "44"]),
    ("A3 seq baselines ml1m", "r9_seqbaselines_ml1m.log", {},
     [PY, "scripts/run_sequential_baselines.py", "--dataset", "ml1m", "--source-run", V3M,
      "--out", RESULTS / "ml1m_lightgcn_v8_sequential_baselines", "--seeds", "42", "43"]),
    ("A4 ctrlrand amazon", "r9_ctrlrand_amazon.log", {},
     [PY, "scripts/run_controlled_randomization.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v8_controlled_randomization"]),
    ("A4 ctrlrand ml1m", "r9_ctrlrand_ml1m.log", {},
     [PY, "scripts/run_controlled_randomization.py", "--dataset", "ml1m", "--source-run", V3M,
      "--out", RESULTS / "ml1m_lightgcn_v8_controlled_randomization"]),
    ("A5 nested amazon", "r9_nested_amazon.log", {},
     [PY, "scripts/run_nested_lambda_tuning.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v8_nested_tuning", "--seeds", "42", "43", "44"]),
    ("A5 nested ml1m", "r9_nested_ml1m.log", {},
     [PY, "scripts/run_nested_lambda_tuning.py", "--dataset", "ml1m", "--source-run", V3M,
      "--out", RESULTS / "ml1m_lightgcn_v8_nested_tuning", "--seeds", "42", "43"]),
    ("B1 factorial amazon", "r9_factorial_amazon.log", {},
     [PY, "scripts/run_selection_factorial.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v8_selection_factorial", "--seeds", "42", "43"]),
    ("B1 factorial ml1m", "r9_factorial_ml1m.log", {},
     [PY, "scripts/run_selection_factorial.py", "--dataset", "ml1m", "--source-run", V3M,
      "--out", RESULTS / "ml1m_lightgcn_v8_selection_factorial", "--seeds", "42"]),
    ("B2 convergence amazon", "r9_convergence_amazon.log", {},
     [PY, "scripts/run_convergence_v2.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v8_convergence_v2", "--max-users", "1000"]),
    ("B2 convergence ml1m", "r9_convergence_ml1m.log", {},
     [PY, "scripts/run_convergence_v2.py", "--dataset", "ml1m", "--source-run", V3M,
      "--out", RESULTS / "ml1m_lightgcn_v8_convergence_v2", "--max-users", "1000"]),
    ("B3 negset draw 1e6 amazon", "r9_negset_draw_1000000_amazon.log", {},
     [PY, "scripts/run_negset_sensitivity.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v6_negset_sensitivity_draw_1000000",
      "--sizes", "50", "100", "500", "--max-users", "1500", "--neg-offset", "1000000"]),
    ("B3 negset draw 2e6 amazon", "r9_negset_draw_2000000_amazon.log", {},
     [PY, "scripts/run_negset_sensitivity.py", "--dataset", "amazon", "--source-run", V3A,
      "--out", RESULTS / "amazon_books_lightgcn_v6_negset_sensitivity_draw_2000000",
      "--sizes", "50", "100", "500", "--max-users", "1500", "--neg-offset", "2000000"]),
    ("B4 maskedfwd amazon 43-46", "r9_maskedfwd_amazon.log", {"COALGAME_DEVICE": "cpu"}, None),
    ("B4 maskedfwd ml1m 43-46", "r9_maskedfwd_ml1m.log", {"COALGAME_DEVICE": "cpu"}, None),
    ("C1 design ablations amazon 43-44", "r9_designabl_amazon.log", {}, None),
    ("C2 design ablations ml1m 43-44", "r9_designabl_ml1m.log", {}, None),
]

for name, log, env, cmd in STAGES:
    print("=" * 70)
    print("STAGE:", name)
    if cmd is None:  # composite stages
        if "maskedfwd amazon" in name:
            jobs = [(["--dataset", "amazon", "--seed", s]) for s in ["43", "44", "45", "46"]]
            base = [PY, "scripts/run_masked_forward_faithfulness.py"]
        elif "maskedfwd ml1m" in name:
            jobs = [(["--dataset", "ml1m", "--seed", s]) for s in ["43", "44", "45", "46"]]
            base = [PY, "scripts/run_masked_forward_faithfulness.py"]
        elif "amazon" in name:
            jobs = [(["--dataset", "amazon", "--seed", s]) for s in ["43", "44"]]
            base = [PY, "scripts/run_design_ablations.py"]
        else:
            jobs = [(["--dataset", "ml1m", "--seed", s]) for s in ["43", "44"]]
            base = [PY, "scripts/run_design_ablations.py"]
        for j in jobs:
            rc = stream(base + j + (["--n-users", "1000"] if "maskedfwd" in name else []), log, env)
            if rc != 0: print("STAGE SUBJOB FAILED rc=", rc)
    else:
        rc = stream(cmd, log, env)
        if rc != 0: print("STAGE FAILED rc=", rc)
print("ALL STAGES FINISHED")


In [ ]:
# Status overview (safe to run any time)
checks = {
 "A1 v7 amazon": RESULTS / "amazon_books_lightgcn_v7_corrected_protocol" / "tables" / "summary_mean_std.csv",
 "A2 matched sweep amazon": RESULTS / "amazon_books_lightgcn_v8_matched_lambda_sweep" / "tables" / "lambda_sensitivity_mean_std.csv",
 "A3 seq baselines amazon": RESULTS / "amazon_books_lightgcn_v8_sequential_baselines" / "summary_mean_std.csv",
 "A3 seq baselines ml1m": RESULTS / "ml1m_lightgcn_v8_sequential_baselines" / "summary_mean_std.csv",
 "A4 ctrlrand amazon": RESULTS / "amazon_books_lightgcn_v8_controlled_randomization" / "controlled_randomization.json",
 "A4 ctrlrand ml1m": RESULTS / "ml1m_lightgcn_v8_controlled_randomization" / "controlled_randomization.json",
 "A5 nested amazon": RESULTS / "amazon_books_lightgcn_v8_nested_tuning" / "nested_tuning.csv",
 "A5 nested ml1m": RESULTS / "ml1m_lightgcn_v8_nested_tuning" / "nested_tuning.csv",
 "B1 factorial amazon": RESULTS / "amazon_books_lightgcn_v8_selection_factorial" / "selection_factorial.csv",
 "B1 factorial ml1m": RESULTS / "ml1m_lightgcn_v8_selection_factorial" / "selection_factorial.csv",
 "B2 convergence amazon": RESULTS / "amazon_books_lightgcn_v8_convergence_v2" / "convergence_v2.csv",
 "B2 convergence ml1m": RESULTS / "ml1m_lightgcn_v8_convergence_v2" / "convergence_v2.csv",
 "B3 negset draw 1e6": RESULTS / "amazon_books_lightgcn_v6_negset_sensitivity_draw_1000000" / "negset_sensitivity.json",
 "B3 negset draw 2e6": RESULTS / "amazon_books_lightgcn_v6_negset_sensitivity_draw_2000000" / "negset_sensitivity.json",
 "C1 design ablations amazon": (RESULTS / "amazon_books_lightgcn_v3_prospective" / "tables" / "design_ablations_seed_44.csv"),
 "C2 design ablations ml1m": (RESULTS / "ml1m_lightgcn_v3_prospective" / "tables" / "design_ablations_seed_44.csv"),
}
done = sum(1 for p in checks.values() if p.exists())
for name, p in checks.items():
    print(f"{'DONE' if p.exists() else '....'} {name}")
mf = {}
for ds in ["amazon_books", "ml1m"]:
    t = RESULTS / f"{ds}_lightgcn_v3_prospective" / "tables" / "masked_forward_faithfulness.csv"
    if t.exists():
        import pandas as pd
        seeds = sorted(pd.read_csv(t)["seed"].unique().tolist()) if "seed" in pd.read_csv(t, nrows=1) else []
        mf[ds] = seeds
print("B4 masked-forward seeds:", mf)
print(f"\n{done}/{len(checks)} stage outputs present.")


In [ ]:
# PUSH — commit + push everything, then paste results back to the agent loop.
print("Run these from the REPO ROOT (next-paper/):")
print("""
git add paper-ideas/CoalGameRec/code/results/journal_runs
git commit -m "round-9 remaining experiments (notebook R9)"
git push origin arena/019fdd75-next-paper
""")
